# GROMACS MD - RunPod

Single-peptide molecular dynamics pipeline for the SPAFESTWDILK / MDM2 manuscript (CHARMM36m force field, 310 K, 0.15 M NaCl, TIP3P water).

**How to use:** edit the 5 variables in Section 2, then run the cells in order.

**Inputs:** put PDB complexes in `/workspace/pdbs/`. Each PDB must contain the target-peptide complex; the script picks the first file whose name contains the peptide sequence.

**Outputs:** results go to `/workspace/runs/<TARGET>_<PEPTIDE>_<DURATION>_CHARMM36m_310K_<DATE>/` including trajectories (.xtc), topology, .xvg analysis files, and a 6-panel summary plot.

For the paper, this notebook was used to run the 17-system 5 ns triage and the 100 ns production runs of the three promoted candidates (SPAFESTWDILK, EFSDLWDNK, FQSWEDLSK). All inputs and outputs are deposited at https://doi.org/10.5281/zenodo.19859571.

## 1. Setup (run once per pod)

In [ ]:
# Install Python dependencies FIRST (avoids ModuleNotFoundError on fresh pods)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "matplotlib", "numpy"], check=True)

import os, shutil, zipfile, re, time, glob
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

GROMACS_VERSION = "2024.4"

# Detect GPU and pick compute capability
smi = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
sm = "80"
if   "H100"  in smi: sm = "90"
elif "A40"   in smi: sm = "86"
elif "4090"  in smi: sm = "89"
elif "3090"  in smi: sm = "86"
elif "A6000" in smi: sm = "86"
print(f"   GPU: {smi.strip()}, SM={sm}")

# Check if the correct GROMACS is already installed
need_install = True
if subprocess.run("which gmx", shell=True, capture_output=True).returncode == 0:
    ver_out = subprocess.getoutput("gmx --version 2>&1")
    if GROMACS_VERSION in ver_out and "CUDA" in ver_out:
        print(f"OK GROMACS {GROMACS_VERSION} with CUDA already installed")
        need_install = False
    else:
        print("GROMACS found but wrong version/no CUDA, recompiling...")

if need_install:
    print(f"Compiling GROMACS {GROMACS_VERSION} with CUDA (10-15 min)...")
    os.system("apt-get update -qq > /dev/null 2>&1")
    os.system("apt-get install -y -qq cmake build-essential wget > /dev/null 2>&1")

    # Download GROMACS source (with explicit error check)
    tarball = f"/tmp/gromacs-{GROMACS_VERSION}.tar.gz"
    srcdir  = f"/tmp/gromacs-{GROMACS_VERSION}"
    os.system(f"rm -rf {srcdir} {tarball}")
    urls = [
        f"https://ftp.gromacs.org/pub/gromacs/gromacs-{GROMACS_VERSION}.tar.gz",
        f"https://ftp.gromacs.org/gromacs/gromacs-{GROMACS_VERSION}.tar.gz",
    ]
    downloaded = False
    for url in urls:
        print(f"  trying {url} ...")
        rc = os.system(f"wget --tries=3 --timeout=60 -O {tarball} {url}")
        if rc == 0 and os.path.exists(tarball) and os.path.getsize(tarball) > 1_000_000:
            downloaded = True
            print(f"  OK, downloaded {os.path.getsize(tarball)/1e6:.1f} MB")
            break
    if not downloaded:
        raise RuntimeError(f"Could not download GROMACS {GROMACS_VERSION} from any mirror")

    rc = os.system(f"cd /tmp && tar xf gromacs-{GROMACS_VERSION}.tar.gz")
    if rc != 0 or not os.path.isdir(srcdir):
        raise RuntimeError(f"Failed to extract {tarball}")
    print(f"  extracted to {srcdir}")

    cmake_ret = os.system(
        f"cd {srcdir} && mkdir -p build && cd build && "
        f"cmake .. -DGMX_GPU=CUDA -DGMX_BUILD_OWN_FFTW=ON "
        f"-DCMAKE_INSTALL_PREFIX=/usr/local -DREGRESSIONTEST_DOWNLOAD=OFF "
        f"-DGMX_CUDA_TARGET_SM={sm} > /tmp/cmake_gromacs.log 2>&1"
    )
    if cmake_ret != 0:
        print("CMAKE FAILED! Log:")
        os.system("tail -30 /tmp/cmake_gromacs.log")
        raise RuntimeError("cmake failed")

    make_ret = os.system(
        f"cd {srcdir}/build && "
        f"make -j$(nproc) > /tmp/make_gromacs.log 2>&1 && "
        f"make install >> /tmp/make_gromacs.log 2>&1"
    )
    if make_ret != 0:
        print("MAKE FAILED! Log:")
        os.system("tail -30 /tmp/make_gromacs.log")
        raise RuntimeError("make failed")

    os.system("ldconfig /usr/local/lib")
    os.system(f"rm -rf {srcdir} {tarball}")

# Mandatory verification
ver_out = subprocess.getoutput("gmx --version 2>&1")
assert GROMACS_VERSION in ver_out, f"Wrong GROMACS version!\n{ver_out[:300]}"
assert "CUDA" in ver_out, f"GROMACS without CUDA!\n{ver_out[:300]}"
print(f"Verified: GROMACS {GROMACS_VERSION} with CUDA")
os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

# CHARMM36m force field
FF_DIR = "/workspace/tools/charmm36-mar2019.ff"
if not os.path.exists(FF_DIR):
    print("Downloading CHARMM36m...")
    os.system("wget -q https://github.com/intbio/gromacs_ff/archive/refs/heads/master.zip -O /tmp/ff.zip")
    with zipfile.ZipFile("/tmp/ff.zip", 'r') as z:
        z.extractall("/tmp/")
    shutil.copytree("/tmp/gromacs_ff-master/charmm36-mar2019.ff", FF_DIR)
    os.system("rm -rf /tmp/ff.zip /tmp/gromacs_ff-master")
    print("   Installed")
else:
    print("CHARMM36m already present")

os.makedirs("/workspace/pdbs", exist_ok=True)
os.makedirs("/workspace/runs", exist_ok=True)
print("\nSetup complete!")

## 2. Configuration

Edit only the 5 variables below.

For the SPAFESTWDILK / MDM2 paper:

**5 ns triage (17 systems):**
- Candidates: `SPAFESTWDILK`, `FESTWDILK`, `HAFPELWNIEK`, `DDFLNSWK`, `EFSDLWDNK`, `DNEFLQDWSK`, `SDLTSFMEEWR`, `FQSWEDLSK`, `QNSFVDLWK`, `ISTAFLNDWDLAK`, `EHFETLWSSVK`
- Hard negatives: `EWSLDQSKF`, `FSNLDKWDE`, `WSADITKFESPL`, `FWELDSTLKLPNEQS`
- Controls: `AAAAAAAAAAAAAAAAA` (poly-Ala), `SQETFSDLAKLLPEN` (p53 W23A)
- Set `SIM_TIME_NS = 5`

**100 ns production (3 promoted):** `SPAFESTWDILK`, `EFSDLWDNK`, `FQSWEDLSK`
- Set `SIM_TIME_NS = 100`

For all systems: `RESIDUES_RECEPTOR = "1-85"` (MDM2 domain), peptide residues adjusted to peptide length (e.g. `86-94` for a 9-mer, `86-97` for a 12-mer, `86-102` for a 17-mer).

In [ ]:
# ============================================================
#   EDIT ONLY THESE 5 VARIABLES
# ============================================================

TARGET            = "MDM2"
PEPTIDE           = "SPAFESTWDILK"
SIM_TIME_NS       = 1
RESIDUES_RECEPTOR = "1-85"
RESIDUES_PEPTIDE   = "86-97"

# ============================================================
from datetime import datetime
DATE = datetime.now().strftime("%d%b%Y").lower()
RUN_ID = f"{TARGET}_{PEPTIDE}_{SIM_TIME_NS}ns_CHARMM36m_310K_{DATE}"
N_STEPS = int((SIM_TIME_NS * 1000) / 0.002)
WORKDIR  = f"/workspace/runs/{RUN_ID}"
MDP_DIR  = f"{WORKDIR}/mdp"
PLOT_DIR = f"{WORKDIR}/plots"
PDB_DIR  = "/workspace/pdbs"
FF_DIR   = "/workspace/tools/charmm36-mar2019.ff"

for d in [WORKDIR, MDP_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

# Find PDB whose filename contains the peptide sequence
PDB_FILE = None
if os.path.exists(PDB_DIR):
    for f in sorted(os.listdir(PDB_DIR)):
        if PEPTIDE.lower() in f.lower() and f.lower().endswith('.pdb'):
            PDB_FILE = f"{PDB_DIR}/{f}"
            break

print("=" * 65)
print(f"  {RUN_ID}")
print("=" * 65)
print(f"  Target:    {TARGET}")
print(f"  Peptide:   {PEPTIDE} ({len(PEPTIDE)} aa)")
print(f"  Duration:  {SIM_TIME_NS} ns ({N_STEPS:,} steps)")
print(f"  Receptor:  res {RESIDUES_RECEPTOR}")
print(f"  Peptide:   res {RESIDUES_PEPTIDE}")
print(f"  PDB:       {os.path.basename(PDB_FILE) if PDB_FILE else 'NOT FOUND'}")
print(f"  Output:    {WORKDIR}")
print("=" * 65)
if not PDB_FILE:
    print(f"\nPut a PDB in {PDB_DIR}/ whose name contains '{PEPTIDE}'")
    if os.path.exists(PDB_DIR):
        for f in sorted(os.listdir(PDB_DIR)): print(f"   {f}")

## 3. MD pipeline

In [ ]:
# ============================================================
# Full pipeline: pdb2gmx -> solvate -> ions -> EM -> NVT -> NPT -> production -> analysis
# Note: the input PDB is copied as-is (no column reformatting) to avoid
# topology corruption that can lead to spurious peptide dissociation.
# ============================================================

os.chdir(WORKDIR)

print("Copying PDB as-is (no reformatting)...")
shutil.copy(PDB_FILE, f'{WORKDIR}/complex.pdb')
n_atoms = sum(1 for l in open(f'{WORKDIR}/complex.pdb') if l.startswith('ATOM'))
n_ter   = sum(1 for l in open(f'{WORKDIR}/complex.pdb') if l.startswith('TER'))
print(f"   Atoms: {n_atoms}, TER: {n_ter}")

# Symlink force field into the working directory
ff_link = f"{WORKDIR}/charmm36-mar2019.ff"
if not os.path.exists(ff_link):
    os.symlink(FF_DIR, ff_link)

# === MDP file generation ===
def write_mdp(path, params):
    with open(path, 'w') as f:
        for k, v in params.items():
            f.write(f"{k} = {v}\n")

C36 = {"cutoff-scheme":"Verlet", "ns_type":"grid", "coulombtype":"PME",
       "rcoulomb":1.2, "rvdw":1.2, "vdwtype":"Cut-off",
       "vdw-modifier":"Force-switch", "rvdw-switch":1.0, "pbc":"xyz"}

write_mdp(f"{MDP_DIR}/ions.mdp", {**C36,
    "integrator":"steep","emtol":1000,"emstep":0.01,"nsteps":50000,"nstlist":1})
write_mdp(f"{MDP_DIR}/em.mdp", {**C36,
    "integrator":"steep","emtol":500,"emstep":0.01,"nsteps":250000,
    "nstxout":500,"nstlog":500,"nstenergy":500,"nstlist":1})

EQ = {**C36, "integrator":"md","dt":0.002,"nstxout-compressed":5000,
      "nstlog":5000,"nstenergy":5000,"constraint_algorithm":"lincs",
      "constraints":"h-bonds","lincs_iter":1,"lincs_order":4,"nstlist":10,
      "pme_order":4,"fourierspacing":0.16,"tcoupl":"V-rescale",
      "tc-grps":"Protein Non-Protein","tau_t":"0.1 0.1","ref_t":"310 310"}

write_mdp(f"{MDP_DIR}/nvt.mdp", {**EQ, "define":"-DPOSRES","nsteps":50000,
    "continuation":"no","pcoupl":"no","gen_vel":"yes","gen_temp":310,"gen_seed":-1})
write_mdp(f"{MDP_DIR}/npt.mdp", {**EQ, "define":"-DPOSRES","nsteps":50000,
    "continuation":"yes","gen_vel":"no","pcoupl":"C-rescale",
    "pcoupltype":"isotropic","tau_p":2.0,"ref_p":1.0,
    "compressibility":"4.5e-5","DispCorr":"EnerPres"})
write_mdp(f"{MDP_DIR}/md.mdp", {**C36,
    "integrator":"md","nsteps":N_STEPS,"dt":0.002,
    "nstxout-compressed":5000,"nstlog":5000,"nstenergy":5000,
    "continuation":"yes","constraint_algorithm":"lincs","constraints":"h-bonds",
    "lincs_iter":1,"lincs_order":4,"nstlist":10,"pme_order":4,"fourierspacing":0.16,
    "tcoupl":"V-rescale","tc-grps":"Protein Non-Protein",
    "tau_t":"1.0 1.0","ref_t":"310 310",
    "pcoupl":"Parrinello-Rahman","pcoupltype":"isotropic",
    "tau_p":2.0,"ref_p":1.0,"compressibility":"4.5e-5",
    "DispCorr":"EnerPres","gen_vel":"no"})
print("   MDP files written")

# === Helpers ===
def run(cmd, name):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"\nERROR in {name}:")
        print(r.stderr[:800])
        with open(f"{WORKDIR}/ERROR_{name}.log",'w') as f: f.write(r.stderr)
        raise RuntimeError(f"Failed: {name}")

def run_analysis(cmd, stdin_text, name):
    r = subprocess.run(cmd, shell=True, input=stdin_text, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  WARN {name} failed: {r.stderr[:200]}")
    else:
        print(f"  OK {name}")

# === Pipeline ===
t0 = time.time()

print("\n[1/6] Topology (pdb2gmx)...")
run("gmx pdb2gmx -f complex.pdb -o processed.gro -water tip3p -ignh -merge all -ff charmm36-mar2019", "pdb2gmx")

print("[2/6] Box + solvent + ions...")
run("gmx editconf -f processed.gro -o box.gro -c -d 1.2 -bt dodecahedron", "editconf")
run("gmx solvate -cp box.gro -cs spc216.gro -o solv.gro -p topol.top", "solvate")
run(f"gmx grompp -f {MDP_DIR}/ions.mdp -c solv.gro -p topol.top -o ions.tpr -maxwarn 5", "grompp_ions")
run("echo SOL | gmx genion -s ions.tpr -o ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15", "genion")

print("[3/6] Energy minimization...")
run(f"gmx grompp -f {MDP_DIR}/em.mdp -c ions.gro -p topol.top -o em.tpr -maxwarn 5", "grompp_em")
run("gmx mdrun -v -deffnm em", "em")

print("[4/6] NVT + NPT equilibration...")
run(f"gmx grompp -f {MDP_DIR}/nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 5", "grompp_nvt")
run("gmx mdrun -v -deffnm nvt -nb gpu -pme gpu -update gpu", "nvt")
run(f"gmx grompp -f {MDP_DIR}/npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 5", "grompp_npt")
run("gmx mdrun -v -deffnm npt -nb gpu -pme gpu", "npt")

print(f"\n[5/6] Production run, {SIM_TIME_NS} ns...")
run(f"gmx grompp -f {MDP_DIR}/md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr -maxwarn 5", "grompp_md")
t_md = time.time()
proc = subprocess.Popen("gmx mdrun -v -deffnm md -nb gpu -pme gpu -update gpu",
    shell=True, stderr=subprocess.PIPE, stdout=subprocess.PIPE, universal_newlines=True)
for line in proc.stderr:
    if "step" in line.lower():
        sys.stdout.write(f"\r  {line.strip()}")
        sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    print("\nMD FAILED!"); raise RuntimeError("MD failed")
print(f"\n  MD complete: {(time.time()-t_md)/60:.1f} min")

print("\n[6/6] Analysis...")
with open('ndx_input.txt','w') as f:
    f.write(f"ri {RESIDUES_RECEPTOR}\nri {RESIDUES_PEPTIDE}\nq\n")
run("gmx make_ndx -f md.tpr -o index.ndx < ndx_input.txt", "ndx")

grp_rec = f"r_{RESIDUES_RECEPTOR}"
grp_pep = f"r_{RESIDUES_PEPTIDE}"

run_analysis("gmx rms -s md.tpr -f md.xtc -n index.ndx -o rmsd_peptide.xvg -tu ns",
             f"{grp_pep}\n{grp_pep}\n", "RMSD")
run_analysis("gmx mindist -s md.tpr -f md.xtc -n index.ndx -od mindist.xvg -on numcont.xvg -d 0.6 -tu ns",
             f"{grp_rec}\n{grp_pep}\n", "Contacts")
run_analysis("gmx hbond -s md.tpr -f md.xtc -n index.ndx -num hbond.xvg -tu ns",
             f'\"{grp_rec}\"\n\"{grp_pep}\"\n', "H-bonds")
run_analysis("gmx gyrate -s md.tpr -f md.xtc -n index.ndx -o gyrate.xvg",
             f'\"{grp_pep}\"\n', "Rg")
run_analysis("gmx rmsf -s md.tpr -f md.xtc -n index.ndx -o rmsf.xvg -res",
             f"{grp_pep}\n", "RMSF")

total = (time.time()-t0)/60
print(f"\n{'='*65}")
print(f"COMPLETE: {RUN_ID} in {total:.0f} min ({total/60:.1f} h)")
print(f"{'='*65}")

## 4. Plots

In [ ]:
os.chdir(WORKDIR)

fig, axs = plt.subplots(2, 3, figsize=(20, 10))
fig.suptitle(f'{TARGET} + {PEPTIDE} -- {SIM_TIME_NS}ns CHARMM36m 310K', fontsize=14, fontweight='bold')

def plot_xvg(ax, fname, title, ylabel, color, div1000=False):
    if os.path.exists(fname):
        data = np.loadtxt(fname, comments=['#','@'])
        if len(data) > 0:
            x = data[:,0]/1000 if div1000 else data[:,0]
            y = data[:,1]
            ax.plot(x, y, color=color, lw=0.5, alpha=0.5)
            w = max(1, len(y)//50)
            if w > 1:
                avg = np.convolve(y, np.ones(w)/w, mode='valid')
                ax.plot(x[:len(avg)], avg, color=color, lw=2)
            m, s = y[len(y)//2:].mean(), y[len(y)//2:].std()
            ax.axhline(m, color='k', ls='--', alpha=0.5)
            ax.text(0.05, 0.95, f'{m:.2f}+/-{s:.2f}', transform=ax.transAxes, va='top', fontsize=9, bbox=dict(fc='white',alpha=0.8))
            ax.set_title(title); ax.set_xlabel('ns'); ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3)
            return m, s
    ax.set_title(title); ax.text(0.5,0.5,'N/A',ha='center',va='center',transform=ax.transAxes)
    return None, None

rmsd_m,rmsd_s = plot_xvg(axs[0,0],'rmsd_peptide.xvg','Peptide RMSD','nm','blue')
cont_m,cont_s = plot_xvg(axs[0,1],'numcont.xvg','Contacts','N','magenta')
hb_m,hb_s     = plot_xvg(axs[0,2],'hbond.xvg','H-bonds','N','teal')
md_m,_        = plot_xvg(axs[1,0],'mindist.xvg','Min distance','nm','red')
rg_m,rg_s     = plot_xvg(axs[1,1],'gyrate.xvg','Radius of gyration','nm','orange',div1000=True)

if os.path.exists('rmsf.xvg'):
    data = np.loadtxt('rmsf.xvg', comments=['#','@'])
    if len(data) > 0:
        axs[1,2].bar(range(len(data)), data[:,1], color='forestgreen', alpha=0.7)
        axs[1,2].set_title('RMSF'); axs[1,2].set_xlabel('Residue'); axs[1,2].set_ylabel('nm')
        axs[1,2].grid(True, alpha=0.3)

plt.tight_layout(rect=[0,0.03,1,0.95])
PLOT_PATH = f"{PLOT_DIR}/{RUN_ID}.png"
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved: {PLOT_PATH}")

verdict = "STABLE" if rmsd_m and rmsd_m < 0.35 else "CHECK" if rmsd_m and rmsd_m < 0.5 else "DISSOCIATION" if rmsd_m else "N/A"
summary = f"""{RUN_ID}
{TARGET} | {PEPTIDE} | {SIM_TIME_NS}ns CHARMM36m 310K
Receptor: {RESIDUES_RECEPTOR} | Peptide: {RESIDUES_PEPTIDE}
RMSD: {f'{rmsd_m:.3f}+/-{rmsd_s:.3f}' if rmsd_m else 'N/A'} | Contacts: {f'{cont_m:.0f}' if cont_m else 'N/A'} | HB: {f'{hb_m:.1f}' if hb_m else 'N/A'} | Rg: {f'{rg_m:.3f}' if rg_m else 'N/A'}
VERDICT: {verdict}"""
print(summary)
with open(f"{WORKDIR}/SUMMARY.txt",'w') as f: f.write(summary)